# Getting all Wastewater study datasets

Here we demonstrate how `mgnipy` can be used to build a cross-study taxonomic dataset for a given biome with rich sample metadata from MGnify and BioSamples in a few lines of code. 

First starting the session with a MGnipy client

In [1]:
from mgnipy import MGnipy

# configure session 
MG = MGnipy(cache_dir='wwtp')

# selecting the studies resource
studies_resource = MG.studies

# helper to see accepted search params for endpoint
studies_resource.describe_endpoint()

List all studies analysed by MGnify

MGnify studies inherit directly from studies (or projects) in ENA.

Supported parameters:
- order: ListMgnifyStudiesOrderType0 | None | Unset
- biome_lineage: None | str | Unset The lineage to match, including all descendant biomes
- has_analyses_from_pipeline: None | PipelineVersions | Unset If set, will only show studies with analyses from the specified MGnify pipeline version
- search: None | str | Unset Search within study titles and accessions
- page: int | Unset Default: 1.
- page_size: int | None | Unset


now using mgnifier to build and then execute the query set

In [2]:
# preparing query set
wwtp_studies = studies_resource(biome_lineage='root:Engineered:Wastewater')
# helper to preview the query set prior to fetch
wwtp_studies.explain()

https://www.ebi.ac.uk/metagenomics/api/v2/studies?biome_lineage=root%3AEngineered%3AWastewater&page=1
https://www.ebi.ac.uk/metagenomics/api/v2/studies?biome_lineage=root%3AEngineered%3AWastewater&page=2
https://www.ebi.ac.uk/metagenomics/api/v2/studies?biome_lineage=root%3AEngineered%3AWastewater&page=3
https://www.ebi.ac.uk/metagenomics/api/v2/studies?biome_lineage=root%3AEngineered%3AWastewater&page=4
https://www.ebi.ac.uk/metagenomics/api/v2/studies?biome_lineage=root%3AEngineered%3AWastewater&page=5
https://www.ebi.ac.uk/metagenomics/api/v2/studies?biome_lineage=root%3AEngineered%3AWastewater&page=6
https://www.ebi.ac.uk/metagenomics/api/v2/studies?biome_lineage=root%3AEngineered%3AWastewater&page=7
https://www.ebi.ac.uk/metagenomics/api/v2/studies?biome_lineage=root%3AEngineered%3AWastewater&page=8


In [3]:
# now actually executing queries
async with MG: 
    # get all 8 pages of records from the list endpoint
    await wwtp_studies.aget_all()
    # enrich the records with study metadata from detail endpoint
    await wwtp_studies.aenrich_details()

Enriching study details: 100%|██████████| 189/189 [00:02<00:00, 65.77it/s]


In [5]:
# taking a look at metdata so far 
wwtp_studies.metadata.to_pandas(expand_nested_dicts=True).head()

,accession,ena_accessions,title,updated_at,downloads,first_accession,biome__biome_name,biome__lineage
0,MGYS00004903,"[ERP112588, PRJEB30157]",EMG produced TPA metagenomics assembly of the ...,2026-05-28T15:46:56.991000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP112588,Wastewater,root:Engineered:Wastewater
1,MGYS00001064,"[ERP009143, PRJEB8105]",The microbial database of activated sludge - D...,2026-05-28T15:46:55.125000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP009143,Activated Sludge,root:Engineered:Wastewater:Activated Sludge
2,MGYS00005006,"[ERP112577, PRJEB30146]",EMG produced TPA metagenomics assembly of the ...,2026-05-28T15:46:50.336000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP112577,Activated Sludge,root:Engineered:Wastewater:Activated Sludge
3,MGYS00000598,"[ERP012631, PRJEB11269]",Pekin duck health with water troughs compared ...,2026-05-28T15:46:49.811000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP012631,Agricultural wastewater,root:Engineered:Wastewater:Industrial wastewat...
4,MGYS00004940,"[ERP112627, PRJEB30196]",EMG produced TPA metagenomics assembly of the ...,2026-05-28T15:46:59.891000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP112627,Activated Sludge,root:Engineered:Wastewater:Activated Sludge


exploring the MGazine of datasets for the studies

In [6]:
# getting the magazine of datasets
MZ = wwtp_studies.datasets
# filtering to taxonomic of interest
filtered_MZ = MZ['v4_1']['Taxonomic assignments SSU']
# with helpers 
taxo_mz = filtered_MZ.taxonomic 

TaxaMGazine containing:
- MGnify pipeline versions: ['v4_1']
- Number of downloads: 114
- Short descriptions: ['Taxonomic assignments SSU']
- Nonempty metadata sets: .mgnify_studies
-----------------------
Next steps: Use `.load()` to initialize.



In [7]:
# lazy loading the datasets
taxo_mz.load()

optionally we use MGnetizers to collect additional information from MGnify given accessions

In [8]:
# collecting run/assembly metadata for given accessions
run_accs = [x for x in taxo_mz.runs_accessions if not x.startswith('ERZ')]
assembly_accs = [x for x in taxo_mz.runs_accessions if x.startswith('ERZ')]
# init mgnetizer to collect 
mnet_run = MG.mgnetizer(resource='run', all_ids=run_accs)
mnet_assembly = MG.mgnetizer(resource='assembly', all_ids=assembly_accs)
# now executing the requests to the detail endpoints
async with MG: 
    await mnet_run.aenrich(limit=None)
    await mnet_assembly.aenrich(limit=None)

Enriching metadata from MGnify: 100%|██████████| 244/244 [00:02<00:00, 97.04it/s] 


In [9]:
# passing the additional metadata to the Mgazine of taxonomic datasets
taxo_mz.mgnify_runs = mnet_run.metadata.to_list() + mnet_assembly.metadata.to_list()

optionally can also use BioSampler helper to collect additional sample metadata from BioSamples

In [11]:
# # the sample accessions to use
# sample_ids = taxo_mz.mgnify_runs.to_pandas()['sample_accession'].unique()
# # init biosampler to collect
# bios = MG.biosampler(sample_ids)
# # actually executing the requests
# async with MG: 
#     await bios.aenrich(limit=None)
# # passing the matadata back to MGazine
# taxo_mz.biosamples_metadata = bios.metadata.to_list(drop_duplicates=True)
# # taking a look 
# print(taxo_mz)

from the TaxaMGazine we can get an annotated dataframe with the observation metadata and taxonomic metadata (i.e., taxonomic ranks)

In [12]:
# convert to annotated dataframe
an_df = taxo_mz.to_anndata()
# demo adding a layer with filled in zeros 
an_df.layers['filled_zeros'] = an_df.to_df().fillna(0)
# exporting to h5ad file 
an_df.obs = an_df.obs.astype(str) #workaround for h5ad export issue with mixed types in obs
an_df.write_h5ad('wwtp_biome.h5ad')

as an example here we demo how to use the dataset in a new script:

In [13]:
import anndata as ad 
# read in data
back = ad.read_h5ad('wwtp_biome.h5ad')
# check it out
back

AnnData object with n_obs × n_vars = 1411 × 11604
    obs: 'experiment_type', 'instrument_model', 'instrument_platform', 'sample_accession', 'study_accession', 'updated_at', 'run_accession', 'reads_study_accession', 'assembly_study_accession', 'assembler_name', 'assembler_version', 'status', 'sample__accession', 'sample__ena_accessions', 'sample__sample_title', 'sample__biome', 'sample__updated_at', 'study__accession', 'study__ena_accessions', 'study__title', 'study__updated_at', 'study__biome.biome_name', 'study__biome.lineage', 'GivenID', 'RunID', 'SRA accession', 'name', 'taxid', 'ENA-CHECKLIST', 'ENA-FIRST-PUBLIC', 'ENA-LAST-UPDATE', 'External Id', 'INSDC center name', 'INSDC first public', 'INSDC last update', 'INSDC status', 'Submitter Id', 'amount or size of sample collected', 'collection date', 'description', 'environment (biome)', 'environment (feature)', 'environment (material)', 'geographic location (country and/or sea)', 'geographic location (latitude)', 'geographic locatio